# Markov Chain–Based Behavioral Predictability Analysis 

To assess how predictable each participant’s behavior was during the Rock–Paper–Scissors game, the authors applied a Markov chain analysis to the behavioral response data. The goal of this analysis was not to model "optimal play", but to quantify the extent to which a participant’s next response could be predicted from their previous responses. Prediction accuracy was used as a direct measure of behavioral predictability. In addition to the **probabilistic modeling**, **descriptive statistical analysis** also plays a significant role in the paper. The goal is to quantify deviations from randomness and to identify structured patterns in participants’ decisions.

The core statistical tool used in the paper is the **first-order Markov chain**, which models sequential dependencies between consecutive responses. Rather than evaluating isolated choices, this approach captures how the probability of a response depends on the immediately preceding action.

The analysis was conducted separately for each participant. For every trial in the experiment, the authors tracked participants’ responses over time and computed how often each response (Rock, Paper, or Scissors) followed a specific response on the previous trial. The resulting prediction accuracy from the stochastic Markov analysis allowed the authors to determine whether participants behaved randomly or followed quasi-systematic patterns during the trials. More on that is covered in the paper resutls reproduction section part.


# Role of Statistical Plots

In addition to the Markov analysis, several statistical visualizations were used in the study to characterize behavioral patterns more comprehensively. These plots serve complementary purposes.

## Behavioral Distribution Plots

To further analyze decision-making behavior, additional plots were used:

- **Outcome distribution (wins, losses, draws)** → evaluates whether game dynamics are balanced  
- **Response frequency (most / mid / least played)** → reveals biases in action selection  
- **Response switching behavior (after win/loss/draw)** → captures sequential strategies such as win–stay or lose–shift  

These are visualized using **raincloud plots** (Figure 1c–e), which combine:
- boxplots (summary statistics)  
- density estimates (distribution shape)  
- individual data points (variability)  

This type of visualization provides a detailed representation of both group-level trends and individual differences.


## Interpretation of the Framework

The statistical analyses described above are interpreted relative to a well-defined baseline of random behavior. In the context of the Rock–Paper–Scissors task, random decision-making implies that each of the three possible responses is selected with equal probability, resulting in a theoretical chance level of \( \frac{1}{3} \approx 33.3\% \). This baseline serves as an important reference point for evaluating whether observed behavioral patterns reflect randomness or structured decision-making.

Within the Markov analysis, prediction accuracy provides a direct measure of behavioral predictability. If participants behave randomly, the model’s ability to predict their next action should not exceed chance level. However, systematic deviations above this baseline indicate that responses are not independent across trials, but instead exhibit sequential dependencies. In other words, participants’ choices are influenced by their previous actions, leading to predictable patterns that can be captured by the transition probabilities of the Markov model.

The complementary statistical analyses further characterize the nature of these deviations from randomness. The distribution of responses across Rock, Paper, and Scissors provides insight into potential choice biases, since under a purely random strategy each option would be selected with equal frequency. This effect is illustrated in Figure 1d of the paper.

Similarly, the analysis of response changes across trials offers insight into dynamic behavioral adjustments, as depicted in Figure 1e. By examining how often participants switch or repeat their responses following specific outcomes (e.g., wins, losses, or draws), it becomes possible to identify structured strategies such as win–stay or lose–shift behavior.

Taken together, these statistical measures provide a comprehensive framework for assessing human behavior in this competitive setting. The combination of Markov-based predictability analysis and descriptive statistical visualization therefore allows for a nuanced understanding of how human decision-making deviates from theoretically optimal randomness.



-----


# Paper Behavioural Reproduction

## Behavioral Data Structure

The behavioral responses used for the Markov analysis are stored in
**TSV** files for each participant pair. Each row
corresponds to a single trial in the experiment and contains timing
information, the responses of both players, their reaction times, and
the outcome of the round.

A simplified example from the original data structure is shown below:

| onset | duration | onset_sample | trial_num | player1_resp | player1_rt | player2_resp | player2_rt | outcome |
|------|--------|-------------|---------|-------------|-----------|-------------|-----------|-------|
| 420.78 | 5 | 861754 | 1 | 2 | 0.025 | 1 | 1.308 | 2 |
| 425.77 | 5 | 871977 | 2 | 3 | 1.51 | 1 | 1.11 | 3 |

The columns relevant for the Markov analysis are the response columns:

- **player1_resp** – response of player 1  
- **player2_resp** – response of player 2  

Responses are encoded numerically:

- **1 = Rock**
- **2 = Paper**
- **3 = Scissors**
- **0 = No response**

For each pair of participants, a number of **480 trials** was recorded and extracted from the **TSV** files in order to be processed as input for the Markov chain algorithm.



## First-Order Markov Chain Model

The behavioral predictability analysis is based on a **first-order Markov chain**, a probabilistic model commonly used to describe sequential processes in which the probability of the next state depends only on the current state.

Formally, the **Markov property** states that the future probability of a random variable in a Markov chain is conditionally independent of all earlier states (probabilities), given the present state. In mathematical terms:

$$
P(X_{t+1} \mid X_t, X_{t-1}, ..., X_1) = P(X_{t+1} \mid X_t)
$$

where   
- $ X_t $ denotes the response at trial $t$  
- $ X_{t+1} $ denotes the response at the next trial  

This property implies that the model **does not consider** the entire history of responses, but only the most recent one when predicting the next action.



# Transition Counts

The first step of the algorithm implementation constructs cumulative counts of response transitions across trials. These counts are stored in a matrix called `prob_data`.

Each row corresponds to a trial and contains cumulative counts of transitions observed up to that trial. The matrix has the following structure:

| Column | Meaning |
|--------|--------|
| 1      | # Rock |
| 2–4    | Rock → (R, P, S) |
| 5      | # Paper |
| 6–8    | Paper → (R, P, S) |
| 9      | # Scissors |
| 10–12  | Scissors → (R, P, S) |

The following code snippet illustrates the iterative update process of the matrix:

```python
if prev_resp == 1:  # Rock
    prob_data[i, 1] += 1
    prob_data[i, 2 + (curr_resp - 1)] += 1

elif prev_resp == 2:  # Paper
    prob_data[i, 5] += 1
    prob_data[i, 6 + (curr_resp - 1)] += 1

elif prev_resp == 3:  # Scissors
    prob_data[i, 9] += 1
    prob_data[i, 10 + (curr_resp - 1)] += 1
```

During each iteration, the counts from the previous trial are copied and
updated according to the current transition.


If the previous response was Rock, Paper, or Scissors, the corresponding
transition counts are incremented.




# Sliding Window Transition Estimation

Rather than computing transition probabilities from the entire response
history, the algorithm used in the paper estimates them using a **sliding window** of
recent trials.

Window sizes between **5 and 100 trials** are tested. These are also included in the main plots in the paper. For each trial
$i$, the transition counts inside the window are computed by subtracting
cumulative counts:

$$
\text{counts}_w = \text{prob\_data}[i - 1] - \text{prob\_data}[i - w]
$$

If the current trial occurs earlier in the experiment and a full window is
not yet available, all transitions observed up to that particular trial are used
instead.

```python
if i < window_size + 1:
    inter = prob_data[i - 1, :]
else:
    inter = prob_data[i - 1, :] - prob_data[i - window_size, :]
```

### Transition Probabilities

The behavior of the Markov chain is fully defined by its **transition probabilities**, which describe the likelihood of moving from one response to another between consecutive trials.

In our code we have for example:

- $P(P|R)$ — probability of playing **Paper after Rock**
- $P(S|P)$ — probability of playing **Scissors after Paper**
- $P(R|S)$ — probability of playing **Rock after Scissors**

These probabilities are given empirically from the observed responses in the studied trails.


Those Transition probabilities are estimated as follows:

$$
P(j \mid i) = \frac{\text{count}(i \rightarrow j)}{\text{count}(i)}
$$

If insufficient data is available:

$$
P = \frac{1}{3}
$$

Here the code snippet for computing the probabilities:

```python
if inter[1] > 0:
    m_prob[0, :] = inter[2:5] / inter[1]
else:
    m_prob[0, :] = 1 / 3
```

Furthermore, the conditional probability of Paper following Rock is calculated as:

$$
P(P|R) =
\frac{\text{Number of transitions } R \rightarrow P}
{\text{Total number of transitions starting from } R}
$$

This estimation approach corresponds to the **maximum likelihood estimate (MLE)** of the transition probabilities given the observed data.



## Transition Matrix

The core of the Markov model and also in the paper implementation is the **transition probability matrix**.
This matrix contains the probabilities of moving from one response to
another and has size (3 x 3).

$$
T =
\begin{bmatrix}
P(R|R) & P(P|R) & P(S|R) \\
P(R|P) & P(P|P) & P(S|P) \\
P(R|S) & P(P|S) & P(S|S)
\end{bmatrix}
$$

Each **row** corresponds to the previous response, while each column
corresponds to the predicted next response**.


If a participant frequently plays **Paper after Rock**, then $P(P\|R)$
will be high. If the participant behaves randomly, all probabilities
approach **1/3**.

Transition probabilities are computed as:

$$
P(P|R) = \frac{\text{Number of times Paper followed Rock}}{\text{Total number of Rock responses}}
$$

## Prediction Rule and Accuracy Computation

Once the transition probabilities are estimated, the model generates
trial-by-trial predictions of the participant’s next response.

For each trial $i$, the algorithm:

1. Identifies the most recent valid response $ X_t $
2. Retrieves the corresponding row of the transition matrix
3. Selects the most likely next response using:

$$
\hat{X}_{t+1} = \operatorname*{argmax}_{x \in \{R,P,S\}} P(x \mid X_t)
$$

```python
last_resp = responses[idx - 1] - 1
probs = m_prob[last_resp]

pred_move = np.argmax(probs)
prob_res[i, 1] = pred_move + 1
prob_res[i, 2] = probs[pred_move]
```

The predicted response is the one with the **highest probability**,
which is obtained using an **argmax** function.

### Handling Missing Responses

In cases where a participant did not respond (coded as 0), the model
falls back to the most recent valid response:

- If $( X_{t-1}) $ is invalid → use $ X_{t-2} $
- If no valid response exists → prediction is skipped

This ensures that predictions are always based on meaningful
behavioral history.

### Accuracy Measure

Prediction performance is evaluated using **classification accuracy**:

$$
\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of predictions}}
$$

Formally:

$$
\text{Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(\hat{X}_i = X_i)
$$

where:
- $ \hat{X}_i$ is the predicted response  
- $ X_i $ is the true response  
- $ \mathbb{I} $ is the indicator function  

Chance-level performance in this task is:

$$
\text{Chance} = \frac{1}{3} \approx 33.3\%
$$

Only trials from index 3 onward are used, ensuring that sufficient
history is available for prediction.




## Group-Level Analysis (Paper Figure 1f)

To evaluate whether behavioral predictability is consistent across participants, in the paper a **group-level analysis** was performed and reproduced by us by aggregating prediction accuracies across all subjects and window sizes. This allowed to determine whether the observed predictability is a robust phenomenon rather than an individual-specific effect.


### Data Aggregation

For each participant (including both players in each pair), the Markov analysis produces a vector of prediction accuracies:

$$
\text{accuracy} \in \mathbb{R}^{|W|}
$$

where $$ W = \{5, 6, ..., 100\} $$ denotes the set of sliding window sizes.

All participants are combined into a matrix:

$$
A \in \mathbb{R}^{N \times |W|}
$$

where:

 * N  = total number of participants (both players pooled)  
*  |W|  = number of window sizes  

Each row corresponds to one participant, and each column corresponds to a specific window size.



```python
group_mean = np.mean(all_participants, axis=0)
group_sem = np.std(all_participants, axis=0) / np.sqrt(len(all_participants))
```


The group-level Markov analysis confirms that participants exhibit systematic and predictable behavior that deviate from the optimal random play strategy. This could be observed in the paper graph but also in our same reproduction form the code. This reproduces the key behavioral finding of the original paper, showing that human decisions in such competitive settings like playing a Rock, Paperm Scissors game become structured rather than random.

## Behavioral Analyses (Figure 1c–e)

To complement the Markov predictability analysis as in the original paper, we reproduced the key behavioral statistics and plots reported in the original paper. These analyses provide insight into how participants actually behave during the game and whether their choices align with random or strategic play.

### Game Outcome Distribution (Fig. 1c)

We first quantified the distribution of game outcomes across all trials for each participant. Specifically, we computed:

- Percentage of wins (for the overall winner)  
- Percentage of losses (for the overall winner)  
- Percentage of draws  

```python
percent_won, percent_lost, percent_drawn = subject.get_winners_outcome_distribution()
```

This analysis serves as a sanity check to ensure that the game dynamics are balanced. In an ideal random setting as we previosly mentioned, one would expect approximately equal proportions of wins, losses, and draws (each close to 33.3%). Some deviations from this distribution may indicate strategic imbalances or behavioral biases.

## Response Bias (Fig. 1d)

Next, we reproduced/examined whether participants exhibited preferences for specific responses (e.g., Rock, Paper, or Scissors). For each participant, we ranked as in the paper responses by frequency and extracted:


- Most frequently played response  
- Moderately played response  
- Least frequently played response  

This allows to assess whether participants follow a uniform random strategy or show systematic biases.

**Observation:**  
Participants consistently favored certain responses, most notably *Rock*, indicating a clear deviation from uniform randomness. Instead of selecting each option with equal probability (1/3), participants displayed skewed choice distributions.

This bias suggests:

- Habit formation or default strategies, for example that *Rock* is most frequently chosen or Scissors is least frequently chosen  
- Cognitive shortcuts in decision-making  
- Potential exploitability by adaptive opponents  



## Response Switching (Fig. 1e)

In Fig. 1e is analyzed how participants adjusted their behavior based on previous outcomes. Specifically, we computed the probability of switching to a different response after:

- A win  
- A loss  
- A draw  


$$
\text{Change} = (X_t \neq X_{t-1})
$$


The expected probability of switching under randomness is:

$$
\frac{2}{3} \approx 66.7\%
$$

Results showed that participants tend to **switch responses more
often than expected**, consistent with heuristic strategies such as:

- win–stay / lose–shift  
- cyclic patterns  


### Raincloud Plot Implementation

All behavioural plots were implemented using our function `eeg_visualization.py`.

The visualization combines:

- Kernel density estimation (distribution shape)
- Boxplot (median + spread)
- Jittered scatter (individual data points)

This closely matches the visualization style used in the original paper.

